In [1]:
import json
import math
import os
import random
import shutil
import time
from datetime import date, timedelta
from pathlib import Path

# CONFIGURATION & CONSTANTS
BASE_DIR = Path("/tmp/flights")
NUM_FILES = 5000
CITY_POOL_SIZE = 150  # K within range [100, 200]
MIN_RECORDS_PER_FILE = 50  # Range [50, 100]
MAX_RECORDS_PER_FILE = 100
DIRTY_PROBABILITY = 0.008  # L within range [0.5%, 1.0%]

# Mock list of cities
CITIES = [f"City_{i:03d}" for i in range(1, CITY_POOL_SIZE + 1)]


def generate_flight_record(origin_city: str) -> dict:
    """Generates a single flight record, occasionally injecting NULLs."""
    destination_city = random.choice([c for c in CITIES if c != origin_city])

    random_days = random.randint(0, 365)
    flight_date = (date.today() - timedelta(days=random_days)).isoformat()

    record = {
        "date": flight_date,
        "origin_city": origin_city,
        "destination_city": destination_city,
        "flight_duration_secs": random.randint(1800, 43200),
        "#_of_passengers_on_board": random.randint(20, 350),
    }

    if random.random() < DIRTY_PROBABILITY:
        fields_to_nullify = random.sample(
            list(record.keys()), k=random.randint(1, 3)
        )
        for field in fields_to_nullify:
            record[field] = None

    return record


def run_phase_1():
    """Phase 1: Data Generation."""
    print("--- Phase 1: Starting Data Generation ---")

    # Wipe existing directory to clear out bad folder structures
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR)

    BASE_DIR.mkdir(parents=True, exist_ok=True)
    current_mmyy = date.today().strftime("%m-%y")

    for i in range(NUM_FILES):
        origin_city = random.choice(CITIES)

        # File directly follows the naming format requirement
        file_path = (
            BASE_DIR / f"{current_mmyy}-{origin_city}-flights_{i+1}.json"
        )

        num_records = random.randint(MIN_RECORDS_PER_FILE, MAX_RECORDS_PER_FILE)
        file_data = [
            generate_flight_record(origin_city) for _ in range(num_records)
        ]

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(file_data, f, indent=2)

    print(
        f"Phase 1 Complete: Generated {NUM_FILES} files directly in {BASE_DIR}.\n"
    )


def run_phase_2():
    """Phase 2: Data Analysis & Cleaning."""
    print("--- Phase 2: Starting Data Analysis & Cleaning ---")
    start_time = time.perf_counter()

    total_records = 0
    dirty_records = 0

    dest_durations = {}
    dest_passengers = {}
    passenger_balance = {city: 0 for city in CITIES}

    # Only process actual files (skip directories if any exist)
    json_files = [p for p in BASE_DIR.glob("**/*.json") if p.is_file()]

    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                records = json.load(f)
            except json.JSONDecodeError:
                continue

        for record in records:
            total_records += 1

            if any(value is None for value in record.values()):
                dirty_records += 1
                continue

            origin = record["origin_city"]
            dest = record["destination_city"]
            duration = record["flight_duration_secs"]
            passengers = record["#_of_passengers_on_board"]

            dest_durations.setdefault(dest, []).append(duration)
            dest_passengers[dest] = dest_passengers.get(dest, 0) + passengers

            passenger_balance[origin] = (
                passenger_balance.get(origin, 0) - passengers
            )
            passenger_balance[dest] = (
                passenger_balance.get(dest, 0) + passengers
            )

    top_25_dests = sorted(
        dest_passengers.items(), key=lambda x: x[1], reverse=True
    )[:25]

    top_25_metrics = []
    for dest, _ in top_25_dests:
        durations = sorted(dest_durations[dest])
        n = len(durations)

        avg_dur = sum(durations) / n if n > 0 else 0
        p95_index = math.ceil(0.95 * n) - 1
        p95_dur = durations[max(0, p95_index)] if n > 0 else 0

        top_25_metrics.append(
            {
                "city": dest,
                "total_passengers": dest_passengers[dest],
                "avg_duration_sec": round(avg_dur, 2),
                "p95_duration_sec": p95_dur,
            }
        )

    max_balance_city = max(passenger_balance.items(), key=lambda x: x[1])
    min_balance_city = min(passenger_balance.items(), key=lambda x: x[1])

    elapsed_ms = (time.perf_counter() - start_time) * 1000

    print("================ ANALYSIS RESULTS ================")
    print(f"Total Records Processed : {total_records}")
    print(f"Total Dirty Records     : {dirty_records}")
    print(f"Total Runtime           : {elapsed_ms:.2f} ms\n")

    print("Top 25 Destination Cities Metrics:")
    print(
        f"{'City':<12} | {'Passengers':<12} | {'Avg Duration (s)':<18} | {'P95 Duration (s)':<16}"
    )
    print("-" * 65)
    for m in top_25_metrics:
        print(
            f"{m['city']:<12} | {m['total_passengers']:<12} | {m['avg_duration_sec']:<18} | {m['p95_duration_sec']:<16}"
        )

    print("\nPassenger Balance Analysis:")
    print(
        f"Max Remaining Passengers : {max_balance_city[0]} ({max_balance_city[1]:+} passengers)"
    )
    print(
        f"Min Remaining Passengers : {min_balance_city[0]} ({min_balance_city[1]:+} passengers)"
    )
    print("==================================================")


# Execute directly in cell
run_phase_1()
run_phase_2()

--- Phase 1: Starting Data Generation ---
Phase 1 Complete: Generated 5000 files directly in /tmp/flights.

--- Phase 2: Starting Data Analysis & Cleaning ---
================ ANALYSIS RESULTS ================
Total Records Processed : 375393
Total Dirty Records     : 3051
Total Runtime           : 543.09 ms

Top 25 Destination Cities Metrics:
City         | Passengers   | Avg Duration (s)   | P95 Duration (s)
-----------------------------------------------------------------
City_021     | 479925       | 22281.63           | 40789           
City_121     | 479795       | 22642.35           | 41192           
City_082     | 479286       | 22673.59           | 41177           
City_057     | 478821       | 22525.84           | 41047           
City_052     | 476796       | 22353.08           | 40780           
City_046     | 476700       | 22407.41           | 41332           
City_005     | 476322       | 22187.0            | 41000           
City_077     | 476121       | 21971.06      